<a href="https://colab.research.google.com/github/dianajuvi89/PACE-Atopic-Dermat-Meta-EWAS/blob/main/Copy_of_genomeCL_day4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Genomics on the command line day 4: more tabular data

## Setting up the workshop
This workshop is organized as a `jupyter` notebook, which allows us to work in an interactive environment to edit and run code in small "blocks", without requiring you to install various packages yourselves. We'll be running it using [Google Colab](https://colab.research.google.com/): either click on the "Open Jupyter Notebook" link on the [landing page for the workshop](https://informatics.fas.harvard.edu/workshops/biotips/), or if you have downloaded the notebook yourself, just upload it to Colab and open it.

Jupyter notebooks have blocks of formatted text ("text" or "markdown" blocks), as well as "code" blocks that contain executable commands that can be run within the notebook. Clicking on the arrow in the upper left of a code block will execute the code in that block.

### Install software

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:09
🔁 Restarting kernel...


In [ ]:
import condacolab
condacolab.check()

!conda install -c bioconda bedtools
!conda install -c bioconda samtools

✨🍰✨ Everything looks OK!
Channels:
 - bioconda
 - conda-forge
Platform: linux-64
Solving environment: / - \ | done

## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - bedtools


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    bedtools-2.31.1            |       h13024bc_3         1.5 MB  bioconda
    ca-certificates-2026.4.22  |       hbd8a1cb_0         128 KB  conda-forge
    certifi-2026.4.22          |     pyhd8ed1ab_0         132 KB  conda-forge
    conda-24.11.3              |  py311h38be061_0         1.1 MB  conda-forge
    openssl-3.6.2              |       h35e630c_0         3.0 MB  conda-forge
    ------------------------------------------------------------
                                           Total:         6.0 MB

The following NEW packages will be INSTALLED:

  bedtools           bioconda/linux-64::bedtools-2.31.1-h13024bc_3 



### Download example data

In [ ]:
%%bash
mkdir -p data_day4
wget -P data_day4/ https://raw.githubusercontent.com/harvardinformatics/biotips/refs/heads/main/data/example.bed
wget -P data_day4/ https://raw.githubusercontent.com/harvardinformatics/biotips/refs/heads/main/data/sv.bed
wget -P data_day4/ https://raw.githubusercontent.com/harvardinformatics/biotips/refs/heads/main/data/reads_vs_reference.mapped.sorted.bam
wget -P data_day4/ https://raw.githubusercontent.com/harvardinformatics/biotips/refs/heads/main/data/reads_vs_reference.mapped.sorted.bam.bai
wget -P data_day4/ https://raw.githubusercontent.com/harvardinformatics/biotips/refs/heads/main/data/example.csv
wget -P data_day4/ https://raw.githubusercontent.com/harvardinformatics/biotips/refs/heads/main/data/example.gff

mkdir -p img
wget -P img/ https://raw.githubusercontent.com/harvardinformatics/biotips/refs/heads/main/img/annotation_coordSystems.png

--2026-04-23 13:51:26--  https://raw.githubusercontent.com/harvardinformatics/biotips/refs/heads/main/data/example.bed
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 409398 (400K) [text/plain]
Saving to: ‘data_day4/example.bed’

     0K .......... .......... .......... .......... .......... 12% 4.01M 0s
    50K .......... .......... .......... .......... .......... 25% 4.90M 0s
   100K .......... .......... .......... .......... .......... 37% 27.0M 0s
   150K .......... .......... .......... .......... .......... 50% 16.9M 0s
   200K .......... .......... .......... .......... .......... 62% 7.86M 0s
   250K .......... .......... .......... .......... .......... 75% 32.7M 0s
   300K .......... .......... .......... .......... .......... 87% 66.7M 0s
   350

## Parsing tabular data with `awk`
Welcome to day 4! We are going to be continuing exploring tabular data files and tools to manipulate them, focusing on bioinformatic types of tabular data such as BED files, which as we discussed last time are a type of interval or annotation file. We'll start by introducing one of the versatile command line tools for working with tables, the `awk` programming language.

Invented in the 1970's, `awk` is a scripting language included in most Unix-like operating systems. It specializes in one-liner programs and manipulating tabular text files, and like most scripting languages it is also capable of various mathematical and logical operations.

In many cases, if you're parsing information from a text file, you could write a Python script... or you could do it with `awk` in a single line! We can't expect to learn the entire language in this one workshop, so we will focus on how we can use `awk` for quick data analysis and manipulation.

### `awk` syntax

`awk` scripts are organized as "action blocks":

`awk 'pattern { action; other action }' input.txt`

This means that `awk` reads `input_file.txt` line by line, and for each line performs both `action` and `other action`. A semi-colon (`;`) is used to de-limit separate actions.

There is also a `pattern` that can be included before an action block, which functions similarly to an `if/then` statement in other programming languages. Meaning that every time that the pattern evaulates to *true*, `awk` will execute the action in the brackets. By default, if no pattern is specified it evaulates 'true' every line, so the action will be taken every line, e.g. the following command:

In [ ]:
%%bash
awk '{print}' data_day4/example.bed | head

We don't specify a pattern to test, so this just `print`s every line of the file. We'll show how we can make use of patterns, but first we need to discuss how `awk` interprets input.

### `awk` records and fields
As mentioned, `awk` takes tabular input (either a file or piped from `stdin`) and goes through it line by line, referring to each line as a **record**.

Each record (line) is then split by column into **fields**, based on some **field separator** (i.e. *delimiter*; which is any whitespace by default). `awk` refers to each field *by number* using the syntax `$N` (dollar sign, "N"), where `N` is the specific numbered column. E.g.:

- `$0` refers to the *entire record* (entire line)
- `$1` refers to the first field (column)
- `$2` refers to the second field (column)
- Etc.

So for example:

In [ ]:
%%bash
awk '{print $1}' data_day4/example.bed | head

This command will print the first column of the BED file, and because we do not specify a pattern before the action block it will print every line.

If we want to print multiple fields, we separate them using commas:

In [ ]:
%%bash
awk '{print $4,$1,$2,$3}' data_day4/example.bed | head

This code prints the fourth column of the BED file, then prints the first three.

To be more specific, what the `,` is actually telling `awk` to do is "print the *output field separator character*"... in other words, in plain language what the above command is saying is:

```
print the 4th column -> print output field separator -> print 1st column -> print output field separator -> print 2nd column -> print output field separator -> print 3rd column
```

#### Types of `awk` variables
One important thing to know about how `awk` handles fields is how it determines what *type* of variable each field is. If you are familiar with other programming languages, often when we first define a variable we need to tell the program if the variable is *numeric* (e.g. an integer like `11`) or a *non-numeric* (e.g. some text string like "bingus"); this is referred to as *explicitly typing* a variable.

`awk` is different in that field variables are *dynamically typed*, meaning that `awk` will attempt to "figure out" the type of a variable, without the need to set type manually. For example, if a field contains only numeric characters, `awk` will assign that field as a number.

For numeric fields in `awk`, we can use all the usual mathematical operators: addition `+`, subtraction `-`, multiplication `*`, division `/`, etc. For example, if we wanted to add `1` to each end coordinate of each interval in our BED file and print the new coordinates:

In [ ]:
%%bash
## Command here
awk '{print $3-$2}' data_day4/example.bed | head

50
79
108
137
166
195
224
253
282
311


In [ ]:
%%bash
awk '{print $1,$2,$3+1,$4}' data_day4/example.bed | head

> **Exercise**:
> Write a command that prints the **length** of each interval in the file `data_day4/example.bed`.

In [ ]:
%%bash
#@title Solution {display-mode: "form"}
awk '{print $3-$2}' data_day4/example.bed | head

However, an important thing to remember is that while automatic typing is convenient, it can come back to bite you! For example, say we wanted to run the command to add `1` to the end coordinate of our BED file but we mixed up the columns:

In [ ]:
%%bash
awk '{print $1+1,$2,$3,$4}' data_day4/example.bed | head

We can see that even though the first column is a character string, `awk` still allows us to add a number to it and does NOT produce an error! This is because `awk` will look at *context* when typing the variable, so if it sees a numeric operator being applied to a field (`$1+1`), it will dynamically convert that field to a numeric. Don't get tripped up and *always look at your data!*

### `awk` field separators
By default, the output field separator is a single whitespace character (i.e. ` `), and the input field separator is any whitespace character (space or a tab). So for example, if we had a comma separated file (CSV file) and tried the same command that we ran earlier:

In [ ]:
%%bash
awk '{print $4,$1,$2,$3}' data_day4/example.csv | head

We can see that it does not work correctly. Can anyone explain what is going on?

<details><summary>Solution</summary>

- `awk` reads in each line and uses a whitespace as a field separator...
- However, there are no whitespaces! So `awk` thinks there is only a single column...
- It tries to print the 4th column, which is blank so it prints nothing...
- It prints the 1st column, which is the entire line...
- It tries to print columns 2 and 3 which are also blank

</details>

This is another example of how an incorrect command will not always produce an error message, and why it is always so important to *check yourself* and *have an expectation what what your data should look like*!

If we want `awk` to correctly process files with field separators other than the default, we need some way to alter its behavior...

#### `awk`'s special variables
`awk` has several special built-in variables that determine how data is read in an processed, as well as store information about the data. There are a number of these variables, but the ones that are most useful to us are:

- `FS`: sets the input `F`ield `S`eparator character
- `OFS`: sets the `O`utput `F`ield `S`eparator character
- `NR`: counts the total `N`umber of `R`ecords processed so far
  - I.e. it counts the *current line number* of each line in the file

We define the field separator variables in a separate action block within our `awk` script. For example, to process our CSV file and *convert* it to a TSV file:    

In [ ]:
%%bash
# the first block defines the input and output field separators
# each definition is its own action, so we separate them with a semicolon
# the second block is the print statement, which prints the fields in the desired order separated by OFS (tabs)
awk '{FS="," ; OFS="\t"} {print $4,$1,$2,$3}' data_day4/example.csv | head

> Reminder: as we discussed last time, in text processing "tabs" are their own special characters, `\t` ("backslash t")

We can see that setting `FS=","` now correctly splits the CSV into fields, and by setting the `OFS="\t"` we are outputing the fields separated by tabs.

However, you likely noticed that the first line of the CSV file is NOT being output correctly...for the first line specifically `awk` is still not properly using the `,` as the field separator. Why??

This is due to a quirk of how `awk` processes input: `awk` reads in the first line of a file, AND THEN executes the first action block of the script, i.e. the block where we define our `FS`! So the first line uses the default whitespace separator, the `FS` block is reads, and then all subsequent lines use the correct `FS=","`.

### `BEGIN` and `END` patterns
Recall that before each action block in an `awk` script we can include a **pattern** statement, which changes `awk`'s behavior to check the pattern each line and actions in the block are only executed when that pattern is *true*. If no pattern is specified the block evaluates to *true* every line and the actions will be executed for every line.

Two built-in patterns to `awk` that are extremely useful are the `BEGIN` and `END` patterns:

- `BEGIN` executes before any lines have been read
- `END` executes after there are no more lines to read

Using the `BEGIN` pattern, we can make it so the first line of a CSV file gets the correct field separator character by defining `FS` in a block with `BEGIN`:

In [ ]:
%%bash
# before any lines are read, set FS and OFS
# print columns 4,1,2,3 for every line, separated by OFS (tabs)
awk 'BEGIN{FS="," ; OFS="\t"} {print $4,$1,$2,$3}' data_day4/example.csv | head

> **Exercise**:
> Convert the BED file `data_day4/sv.bed` to CSV format. Save it as a file `data_day4/sv.csv`

In [ ]:
%%bash
## Command here
awk 'BEGIN{FS="\t" ; OFS=","} {print $1,$2,$3,$4}' data_day4/sv.bed > data_day4/sv.csv
head data_day4/sv.csv

chr1,1791,1832,DUP
chr1,5032,5111,INS
chr1,14424,14466,INV
chr1,27335,27378,INS
chr1,42168,42212,CNV
chr1,55079,55124,BND
chr1,68471,68517,DEL
chr1,82946,82993,DUP
chr1,95857,95905,INV
chr1,110690,110739,INS


In [ ]:
%%bash
#@title Solution {display-mode: "form"}
awk 'BEGIN{FS="\t"; OFS=","} {print $1,$2,$3,$4}' data_day4/sv.bed > data_day4/sv.csv

head data_day4/sv.csv

> **Exercise**:
> Write an `awk` command that prints the number of lines in the file `data_day4/example.fastq`, divided by 4.
> Note: this is a quick way to count the number of entries in a FASTQ file without havign to worry about the quality score line!

In [ ]:
%%bash
## Command here
awk 'END{print NR}' data_day4/example.bed
awk 'END{print NR/4}' data_day4/example.bed

18000
4500


In [ ]:
%%bash
#@title Solution {display-mode: "form"}

awk 'END{print NR/4}' data_day4/example.fastq

### Using custom patterns
`BEGIN` and `END` are predefined by `awk` and are super useful, but we can also define our own patterns to pair with action blocks! A pattern is a logical expression that evaluates to *true* or *false*. As before, the pattern goes directly before the action block it is testing. This can be a boolean logical operation, e.g.:

- `==`: equal to
  - Works for numbers AND strings, e.g. `$3 == 0` or `$1 == "string"`
- `!=`: not equal to
- `>`/`>=`: greater than/greater than or equal to
- `<`/`<=`: less than/less than or equal to

Multiple logical statements can be joined using `&&` (AND), `||` (OR).

E.g.:

In [ ]:
%%bash
# print line where the first column is "chr2"
awk '$1 == "chr2" {print $0}' data_day4/example.bed | head # if the first column is equal to "chr2", print all the columns of the file or extract all the information -->{print $0}

chr2	2096	2204	INS
chr2	6288	6512	INV
chr2	10480	10820	DUP
chr2	14672	15128	DEL
chr2	3377	3949	TRA
chr2	7569	8257	BND
chr2	11761	12565	CNV
chr2	15953	16873	INS
chr2	4658	5694	INV
chr2	8850	10002	DUP


In [ ]:
%%bash
# print lines where the second column is greater than 1000 and less than 2000
awk '$2 > 1000 && $2 < 2000 {print $0}' data_day4/example.bed | head

A pattern can also be a regex matching statement, i.e. "does the field or line contain this pattern?" The syntax for this looks like:

- `X ~ /pattern/`: `X` matches `pattern`
- `X !~ /pattern/`: `X` does not match `pattern`

E.g.:

In [ ]:
%%bash
# print lines where the fourth column contains "DUP"
awk '$4 ~ /DUP/ {print $0}' data_day4/example.bed | head

chr1	3144	3281	DUP
chr2	10480	10820	DUP
chr3	2329	2872	DUP
chr4	9665	10411	DUP
chr1	17001	17950	DUP
chr2	8850	10002	DUP
chr3	16186	17541	DUP
chr4	8035	9593	DUP
chr1	15371	17132	DUP
chr2	22707	24671	DUP


> **Exercise**:
> Write an `awk` command that outputs lines in the file `data_day4/example.bed` contains the string `DUP`, and adjusts their coordinates by subtracting `1` from the start.

In [ ]:
%%bash
# Command here
awk '$4 ~ /DUP/ {print $1,$2-1,$3,$4}' data_day4/example.bed | head
awk '$4 ~ /DUP/ {$2=$2-1;print $0}' data_day4/example.bed | head


chr1 3143 3281 DUP
chr2 10479 10820 DUP
chr3 2328 2872 DUP
chr4 9664 10411 DUP
chr1 17000 17950 DUP
chr2 8849 10002 DUP
chr3 16185 17541 DUP
chr4 8034 9593 DUP
chr1 15370 17132 DUP
chr2 22706 24671 DUP
chr1 3143 3281 DUP
chr2 10479 10820 DUP
chr3 2328 2872 DUP
chr4 9664 10411 DUP
chr1 17000 17950 DUP
chr2 8849 10002 DUP
chr3 16185 17541 DUP
chr4 8034 9593 DUP
chr1 15370 17132 DUP
chr2 22706 24671 DUP


In [ ]:
%%bash
#@title Solution {display-mode: "form"}
awk '$4 ~ /DUP/ {print $1,$2-1,$3,$4}' data_day4/example.bed | head

>**Exercise**: write a command that outputs the length of each `INS` interval in the file `data_day4/example.bed`

In [ ]:
%%bash
# Command here
awk '$4 ~ /INS/ {print $3-$2}' data_day4/example.bed | head

108
311
514
717
920
1123
1326
1529
1732
1935


In [ ]:
%%bash
#@title Solution {display-mode: "form"}
awk '$4 ~ /INS/ {print $3-$2}' data_day4/example.bed | head

### Custom `awk` variables
So far we've been using built-in `awk` variables for the fields and records, but like any programming language we can define our own variables. In the `BEGIN` block, we can name our variables and give them an initial value. Like with field variables, they are dynamically typed, so we don't have to specify whether they are numeric or strings.

For example, let's define a variable called `sum` and set it equal to `0`:

In [ ]:
%%bash
awk 'BEGIN{sum=0} {print sum}' data_day4/example.bed | head

0
0
0
0
0
0
0
0
0
0


With this command, we are setting `sum` at the start, but not updating its value, so it prints the same value every line. Let's make another variable called `len`, and for every line we'll set `len` as the length of the interval, add the `len` to the `sum` and print the running sum:

In [ ]:
%%bash
awk 'BEGIN{sum=0; len=0} {len=$3-$2; sum=sum+length; print sum}' data_day4/example.bed | head

Let's add one more piece of logic, and make it so instead of printing the value of `sum` with every line, have it so that only the final value of `sum` is printed:

In [ ]:
%%bash
awk 'BEGIN{sum=0; len=0} {len=$3-$2; sum=sum+length} END{print sum}' data_day4/example.bed

> **Exercise**:
> Write an `awk` script that output the average length of the intervals in `data_day4/example.bed`

In [ ]:
%%bash
# Command here
awk 'BEGIN{sum=0; len=0} {len=$3-$2; sum=sum+len} END{print sum/NR}' data_day4/example.bed


1023.69


In [ ]:
%%bash
#@title Solution {display-mode: "form"}
awk 'BEGIN{sum=0; len=0} {len=$3-$2; sum=sum+len} END{print sum/NR}' data_day4/example.bed

1023.69


## Practical pipelines: finding regions of zero coverage
Before we discuss our final bioinformatic file type, we are going to try to put together everything we've learned across the workshops so far to accomplish a practical kind of task that you might run into in your own research! Imagine that you have assembled a genome, and want to identify regions that might be mis-assembled. As we discussed, one common way to do this is to look at *read coverage*, as regions where no reads align can indicate incorrect assembly. You map your reads agianst your assembly and generate a BAM alignment file. What are your next steps?

When breaking down a task like this, I like to envision what my starting data looks like, and what I want my end data to look like:

> Starting from: a sorted and indexed BAM file of reads aligned against a genome: `data_day4/reads_vs_reference.mapped.sorted.bam`
>
> End Goal: a BED file with the regions of zero depth of coverage in our genome

**Discussion**: let's make an outline of each step of our pipeline, along with the tool(s) we need for each step. Don't worry about syntax just yet, focus on the big picture approach.


<details><summary>Solution</summary>

1. Get depth at each position from the BAM file using `samtools depth`; BAM -> TSV
2. Convert the depth TSV file to a BED file using `awk`
3. Keep only bases with zero coverage using `awk`
4. Merge adjacent zero coverage bases together to create intervals using `bedtools merge`


Now that we have an idea of the overall flow of the pipeline, let work on implementing each step, starting with calculating the depth from the BAM file:

</details>

In [ ]:
%%bash
#@title Solution {display-mode: "form"} "-a function reports all bases where there is/there is no coverage"
samtools depth -o data_day4/reads_vs_reference.depth.tsv -a data_day4/reads_vs_reference.mapped.sorted.bam

<details><summary>Solution</summary>

For the next step, we need to go from the TSV produced by `samtools depth` to a BED file, and also pull out bases that are zero depth. We could do this in two steps using two `awk` commands, but let's try writing it in a single `awk` command!

Let's break down what we need the command to do:
- Print the chromosome name (column 1) from the TSV
- Take the single coordinate (column 2) from the TSV and make it a start-end coordinate for the BED
  - Remember that BED coordinates are 0-based, we we'll want to take the coordinate from the TSV and subtract 1 to get the start for the BED
- Only do this for lines in the TSV that have `0` in the third column
  - Most elegant way to do this would be to use a pattern that is true when `$3==0`
- Just for completeness:
  - Add a `BEGIN` block setting the field separators
  - Add an optional 4th column to the BED with the depth, just to check ourselves

</details>

In [ ]:
%%bash
#@title Solution {display-mode: "form"}
awk 'BEGIN{FS="\t"; OFS="\t"} $3==0{print $1,$2-1,$2,$3}' data_day4/reads_vs_reference.depth.tsv > data_day4/zero_depth.bed

In [ ]:
%%bash
#@title Solution {display-mode: "form"}
awk 'BEGIN{FS="\t"; OFS="\t"} $3==0{print $1,$2-1,$2,$3}' data_day4/reads_vs_reference.depth.tsv > data_day4/zero_depth.bed

In [ ]:
%%bash
#@title Solution {display-mode: "form"}
awk 'BEGIN{FS="\t"; OFS="\t"} $3==0{print $1,$2-1,$2,$3}' data_day4/reads_vs_reference.depth.tsv > data_day4/zero_depth.bed

<details><summary>Solution</summary>

Now we have a BED file, but each interval is only a single base, whereas we want intervals to be entire "blocks" of zero depth. If we look at the documentation for `bedtools merge`, we see that it merges intervals that directly bookend each other, which is exactly what we want. The data must be sorted as well, which we can do either with default `sort` or using `bedtools sort`.

We didn't go over this when we discussed `merge`, but just to avoid making an uncessecary intermediary file we are going to pipe straight from `sort` to `bedtools merge`. You could also make a `sorted.depth.tsv` file first and use that as input if you prefer!

</details>

In [ ]:
%%bash
#@title Solution {display-mode: "form"}

# when piping to merge, we specify the input file (-i) as 'stdin'
sort -k1,1 -k2,2n data_day4/zero_depth.bed | bedtools merge > data_day4/zero_depth_merged.bed

## More interval files: GFF/GTF format
The final bioinformatic file format we will discuss is GFF (`G`eneral `F`eature `F`ormat). They are similar to BED files in that they are *tab-separated tabular data that defines genomic intervals*, with each row defining an interval (aka "feature" or "annotation"). Like with BED files these intervals can represent a variety of things:

- Introns
- Exons
- Repeat regions
- Binding sites
- Heterochromatin regions
- Etc.

However, compared to BED files GFF files are more complex, and contain additional information, requiring 9 columns instead of 3:

1. Chromosome or assembly scaffold ID: The sequence name in the genome assembly file
2. Annotation source: The name of the data source or program that annotated this feature
3. Feature type: A categorical name for the type of feature defined in this row (e.g. "gene", "transcript", "exon")
4. Feature start coordinate: The start position of the feature defined in this row
5. Feature end coordinate: The end position of the feature defined in this row
6. Score: The score of the feature if quality is assessed during annotation, otherwise `.`
7. Strand: Either `+` (forward strand) or `-` (reverse strand)
8. Frame: For coding exons, this indicates the frame as either `0`, `1`, or `2`
9. Attribute: A semi-colon separated list of any other information related to the feature defined in this row

```
head data_day4/example.gff
```

As a note, a related format is the GTF (`G`eneral `T`ransfer `F`ormat), which is very similar to GFF but is more restricted in terms of what kinds of intervals it can represent. We'll focus on GFF for today, as if you understand GFF you'll understand GTF.

### GFF coordinate system
We can see that just like BED files, GFF files have columns for interval coordinates (chromosome, start, end)...however, unlike BED files GFF uses **1-based fully closed coordinates**.

![1 vs 0 coordinates](https://github.com/harvardinformatics/biotips/blob/main/img/annotation_coordSystems.png?raw=1)

This means that chromosomes start numbering at *position 1* (instead of zero), and *both* the start position and the end position are included in the interval. So e.g. bases 2 thru 5 (4 bases total) of an assembly in a 1-based system would be `chr:2-5`, whereas it would be `chr:1-5` in 0-based coordinates.

This is a little more intuitive to read, though it does mean that calculating feature length is less straight-forward:

```
# If end position - start position, gives incorrect size:
5 - 2 = 3bp != 4bp (incorrect)

# Need to add 1 to length:
(5 - 2) + 1 = 4bp (correct)
```

In addition, features that are only a single base pair long (e.g. chr 1, 5th base) are represented as `chr1:5-5` in a 1-based system, which can cause problem for some downstream applications.

As mentioned before, it is important to not mix up 0-based and 1-based coordinate systems and to know which files use which, as it is easy to end up with "off by one" errors.


Due to their added complexity, GFF files are a little more difficult to work with compared to BED files, and often we will want to use specialized programs (e.g. [gffread](https://github.com/gpertea/gffread), [gffutils](https://github.com/daler/gffutils), many others). However, GFF files at their core are just another kind of tabular data, and we can use all the same command line tools that we have been discussing to manipulate them.

> **Exercise**:
> Write a command that converts `data_day4/example.gff` to BED format.


In [ ]:
%%bash
# Command here
awk 'BEGIN{OFS="\t"} {print $1,$4-1,$5}' data_day4/example.gff > data_day4/example_gff_to_bed.bed | head

In [ ]:
%%bash
#@title Solution {display-mode: "form"}
awk 'BEGIN{FS="\t"; OFS="\t"} {print $1,$4-1,$5}' data_day4/example.gff | head

> **Exercise**:
> Write a command that outputs the length of each interval in the file `data_day4/example.gff`

In [ ]:
%%bash
# Command here
awk 'BEGIN{FS="\t"} {print $5-4+1}' data_day4/example.gff | head

2397
1697
1167
1161
2337
1778
1307
1307
1301
1301


In [ ]:
%%bash
#@title Solution {display-mode: "form"}
awk 'BEGIN{FS="\t"} {print $5-$4+1}' data_day4/example.gff | head

### GFF attributes
Most of the columns in GFF files are fairly self-explanatory; however, the final 9th column which contains **attribute information** is important to explain further, as it is what gives GFF files their added complexity.

The 9th column is a semicolon-separated list of key-value pairs, with the syntax `key1=valueA;key2=valueB...`. Each key must be unique to other keys, e.g. you cannot have two `key1`s. Keys are used to store information about the interval: name, identifiers, metadata, relationships (discussed below), etc. The attribute column is not standardized, so depending on the source of the GFF file you may have different keys.

#### GFF feature hierarchies
While attributes are generally speaking not standardized, one exception is GFF files containing *gene annotation information*, a super common use for GFF! If we consider the biology, there is important relationship information between the intervals in a gene annotation: an exon isn't just a generic interval, it "belongs" to a specific transcript mRNA, which in turn "belongs" to a specific gene. In other words, features are *nested* within each other. This is referred to as a **parent-child hierarchy**, where certain features are "children" (or "grandchildren") of "parent" features.

```
gene
  |__mRNA
       |__UTR
       |__CDS
       |__exon 1
       |__exon 2
```

To keep track of these relationships, there are two specific keys:
- `ID=`
- `Parent=`

The value for `ID` must be a UNIQUE identifier that can be used to refer back to a specific feature, while the `Parent` is the unique ID of the parent feature. Note that the `Parent` ID is only one level "above" the feature; e.g. the parent of an exon would be an mRNA, the parent of the mRNA would be a gene, and the gene would not have a parent. Also, sometimes "grandchild" level annotations (exons, CDS, UTRs) will not have an `ID` key, as they have no child features...in other words, a feature only needs a `ID` if it has linked child features.

```
gene1            ================================    ID=gene1
mRNA1            ================================    ID=mRNA1;Parent=gene1
five_prime_UTR   ==                                  Parent=mRNA1
CDS1               ==....=====...........==          Parent=mRNA1
three_prime_UTR                            ======    Parent=mRNA1
mRNA2            ================================    ID=mRNA2;Parent=gene1
exon             ====                                Parent=mRNA2
CDS2               ==....................==          Parent=mRNA2
exon                                     ========    Parent=mRNA2
```

When working with this kind of GFF files it is very important to keep this `ID` + `Parent` structure intact, as downstream programs will depend upon it.

> **Exercise:**
> Write a command that counts the number of mRNAs (i.e. transcripts) in the file `data_day4/example.gff`

In [ ]:
%%bash
# Command here

In [ ]:
%%bash
#@title Solution {display-mode: "form"}
awk 'BEGIN{FS="\t"} $3=="mRNA"{print $0}' data_day4/example.gff | wc -l

> **Exercise**:
> Write a command that calculates the average length of mRNAs in `data_day4/example.gff`

In [ ]:
%%bash
# Command here

In [ ]:
%%bash
#@title Solution {display-mode: "form"}
awk 'BEGIN{FS="\t"; sum=0; len=0; num=0} $3=="mRNA"{len=$5-$4+1; sum=sum+len; num=num+1} END{print sum/num}' data_day4/example.gff

### `bedtools` and GFF files
One last important thing to mention regarding GFF files is that (despite the name) `bedtools`, which we discussed in the last workshop, ALSO works with GFF/GTF files (and other interval files as well)! The program is able to take input and "guess" what format it is, and it will use the correct coordinate system for BEDs vs GFFs.

Let's revisit one of the functions we discussed last time, `bedtools intersect`, which takes two interval files (`A` and `B`) and by default outputs any intervals in `A` that have an overlap with an interval in `B`.

Using the BED file that contains intervals for the structural variants (SVs) in our genome, we want to figure out which *gene annotations* in our GFF file overlap with SVs. We want to ONLY count the SVs that have at least 90% of their interval overlapping a gene.

First, we need to subset the GFF file to only take the gene annotations:

In [ ]:
%%bash
awk 'BEGIN{OFS="\t"} $3=="gene" {print $0}' data_day4/example.gff > data_day4/genes.gff | less -S

Let's look at the `bedtools intersect` documentation to figure out what arguments to the program we need. Our `A` file is `data_day4/genes.gff` and our `B` file is `data_day4/sv.bed`.
- We want to report the entire gene interval if an SV overlaps it, so we need the `-wa` argument
- To only report overlaps when > 90% of the SV intersects, we use `-F 0.9`

In [ ]:
%%bash
bedtools intersect -wa -F 0.9 -a data_day4/genes.gff -b data_day4/sv.bed | wc -l

119
